In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")


In [3]:
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [4]:
# CONFIGURATION
DATA_ROOT = r"/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

# MILESTONE 1

In [5]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path):
            print(f"Path missing! : {genre_path}")
            continue

        # 1. Get all unique song directories
        song_dirs = sorted([d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))])

        valid_songs = []
        for song_dir in song_dirs:
            song_full_path = os.path.join(genre_path, song_dir)

            # CHECK: Completeness & Corruption
            is_valid = True
            temp_paths = {}
            for stem in STEMS:
                stem_file_path = os.path.join(song_full_path, stem)

                # Check if file exists and is larger than 4KB
                if not os.path.exists(stem_file_path) or os.path.getsize(stem_file_path) < 4096:
                    is_valid = False
                    break
                temp_paths[stem.replace('.wav', '')] = stem_file_path

            if is_valid:
                valid_songs.append(temp_paths)

        # 2. Stratified Shuffle Split (per genre)
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]

        # -------------------------------------------------------------------------

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list, genre_key):
            for song in song_list:
                for stem_key, path in song.items():
                    target_dict[genre_key][stem_key].append(path)

        add_to_dict(train_dataset, train_songs, genre)
        add_to_dict(val_dataset, val_songs, genre)

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

In [6]:
import os

def calculate_dataset_stats(root_dir, val_split=0.17, seed=42):
    # Constants for the questions
    KB = 1024
    MB = 1024 * 1024

    corrupted_threshold = 4 * KB
    lower_size_limit = 5.0491 * MB
    upper_size_limit = 5.0493 * MB

    # Counters
    corrupted_count = 0
    less_than_lower = 0
    greater_than_upper = 0

    # Dictionary to store valid song counts per genre for the split question
    genre_valid_counts = {g: 0 for g in GENRES}

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path): continue

        song_dirs = [d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))]

        for song_dir in song_dirs:
            song_full_path = os.path.join(genre_path, song_dir)
            song_stems_valid = True

            for stem in STEMS:
                stem_path = os.path.join(song_full_path, stem)
                if os.path.exists(stem_path):
                    size = os.path.getsize(stem_path)

                    # Q1 & Q2 Logic
                    if size < corrupted_threshold:
                        corrupted_count += 1
                        song_stems_valid = False # Song is invalid if any stem is corrupted
                    if size < lower_size_limit:
                        less_than_lower += 1
                    if size > upper_size_limit:
                        greater_than_upper += 1
                else:
                    song_stems_valid = False

            if song_stems_valid:
                genre_valid_counts[genre] += 1

    # Calculations for Questions
    q1_answer = corrupted_count + less_than_lower
    q2_answer = abs(greater_than_upper - less_than_lower)

    # Q3 Logic: Reggae Train vs Country Val
    reggae_total = genre_valid_counts['reggae']
    reggae_train = int(reggae_total * (1 - val_split))

    country_total = genre_valid_counts['country']
    country_val = country_total - int(country_total * (1 - val_split))

    q3_answer = abs(reggae_train - country_val)

    print(f"Q1 (Corrupted + <5.0491MB): {q1_answer}")
    print(f"Q2 (Abs Diff >5.0493MB and <5.0491MB): {q2_answer}")
    print(f"Q3 (Abs Diff Reggae Train Drum vs Country Val Vocal): {q3_answer}")

calculate_dataset_stats(DATA_ROOT)

Q1 (Corrupted + <5.0491MB): 1256
Q2 (Abs Diff >5.0493MB and <5.0491MB): 1072
Q3 (Abs Diff Reggae Train Drum vs Country Val Vocal): 66


In [7]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    all_tasks = []
    for genre, stems in dataset_dict.items():
        for stem_name, paths in stems.items():
            for path in paths:
                all_tasks.append((genre, stem_name, path))

    for genre, stem_name, file_path in tqdm(all_tasks, desc="Analyzing Silences"):
        y, _ = librosa.load(file_path, sr=sr)
        total_duration = librosa.get_duration(y=y, sr=sr)

        intervals = librosa.effects.split(y, top_db=top_db)

        silence_type = []
        max_silence = 0.0

        if len(intervals) == 0:
            max_silence = total_duration
            silence_type.append("FULL")
        else:
            intervals_sec = librosa.frames_to_time(intervals, sr=sr)

            start_silence = intervals_sec[0][0]
            if start_silence >= threshold_sec:
                silence_type.append("START")

            end_silence = total_duration - intervals_sec[-1][1]
            if end_silence >= threshold_sec:
                silence_type.append("END")

            mid_silence = 0
            for i in range(len(intervals_sec) - 1):
                gap = intervals_sec[i+1][0] - intervals_sec[i][1]
                mid_silence = max(mid_silence, gap)

            if mid_silence >= threshold_sec:
                silence_type.append("MIDDLE")

            max_silence = max(start_silence, end_silence, mid_silence)

        if max_silence >= threshold_sec:
            records.append({
                "Genre": genre,
                "Stem": stem_name,
                "Duration": round(total_duration, 2),
                "Max_Silence_Sec": round(max_silence, 2),
                "Silence_Location": ", ".join(silence_type),
                "File_Path": file_path
            })

    df = pd.DataFrame(records)
    return df


df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

if not df_silence.empty:
    pivot_analysis = df_silence.pivot_table(
        index='Genre',
        columns='Stem',
        values='File_Path',
        aggfunc='count',
        fill_value=0
    )
    print("\n--- Silence Count Analysis (Genre vs Stem) ---")
    print(pivot_analysis)
else:
    print("No long silences found matching the criteria.")

Analyzing Silences: 100%|██████████| 3320/3320 [07:02<00:00,  7.85it/s]


--- Silence Count Analysis (Genre vs Stem) ---
Stem       bass  drums  other  vocals
Genre                                
blues        73     82     75      83
classical    76     61     77      72
country      78     83     56      83
disco        81     83     74      83
hiphop       80     82     80      83
jazz         82     81     82      76
metal        64     74     28      83
pop          81     83     71      82
reggae       83     83     83      82
rock         77     83     57      83


In [8]:
def answer_silence_questions(df):
    # Q4: Total number of sound files having silence >= 5s
    q4_total = len(df)

    # Q5: Total number of sound tracks in Vocals where silence >= 5s
    q5_vocals = len(df[df['Stem'] == 'vocals'])

    # Q6: Average Silence Length in Vocals
    q6_avg_vocal_silence = df[df['Stem'] == 'vocals']['Max_Silence_Sec'].mean()

    # Q7: Total number of drums in jazz where silence >= 5s
    q7_jazz_drums = len(df[(df['Genre'] == 'jazz') & (df['Stem'] == 'drums')])

    # Q8: Total drums in jazz where silence >= 5s AND location is ONLY middle
    q8_jazz_drums_mid = len(df[
        (df['Genre'] == 'jazz') &
        (df['Stem'] == 'drums') &
        (df['Silence_Location'] == 'MIDDLE')
    ])

    # Q9: Total drums in jazz where silence >= 5s AND Max_Silence_Sec >= 10
    q9_jazz_drums_10s = len(df[
        (df['Genre'] == 'jazz') &
        (df['Stem'] == 'drums') &
        (df['Max_Silence_Sec'] >= 10)
    ])

    print(f"Q4 Total Files: {q4_total}")
    print(f"Q5 Total Vocals: {q5_vocals}")
    print(f"Q6 Avg Vocal Silence: {q6_avg_vocal_silence:.2f}")
    print(f"Q7 Jazz Drums: {q7_jazz_drums}")
    print(f"Q8 Jazz Drums (Middle Only): {q8_jazz_drums_mid}")
    print(f"Q9 Jazz Drums (>=10s): {q9_jazz_drums_10s}")

answer_silence_questions(df_silence)

Q4 Total Files: 3063
Q5 Total Vocals: 810
Q6 Avg Vocal Silence: 2701.83
Q7 Jazz Drums: 81
Q8 Jazz Drums (Middle Only): 24
Q9 Jazz Drums (>=10s): 81


In [9]:
import numpy as np
import librosa

def analyze_rock_mix(dataset_dict, genre='rock', song_idx=0):
    # 1. Get paths for the first song in 'rock'
    # The dictionary structure is {genre: {stem: [paths]}}
    stem_paths = {stem: dataset_dict[genre][stem][song_idx] for stem in STEM_KEYS}

    # 2. Load and Combine Stems
    mix_sample = None
    for stem_name, path in stem_paths.items():
        y, _ = librosa.load(path, sr=SR)
        if mix_sample is None:
            mix_sample = y
        else:
            # Summing signals creates the mix
            mix_sample += y

    # Q10: Length of the mix sample
    sample_length = len(mix_sample)

    # Q11: RMS Amplitude
    # Using the standard formula: np.sqrt(np.mean(y**2))
    rms_amplitude = np.sqrt(np.mean(mix_sample**2))

    # Q12: Peak Normalization
    max_peak = np.max(np.abs(mix_sample))

    print(f"Q10 Length: {sample_length}")
    print(f"Q11 RMS Amplitude: {rms_amplitude:.2f}")
    print(f"Q12 Raw Peak Value: {max_peak:.2f}")

    return mix_sample

# Execution
mix = analyze_rock_mix(tr)

Q10 Length: 661794
Q11 RMS Amplitude: 0.11
Q12 Raw Peak Value: 0.74
